# TinyHybridNet + QAT: Quantization-Aware Training on Tiny ImageNet

## Overview
This notebook demonstrates **quantization-aware training (QAT)** on a hybrid efficient CNN architecture.

### Architecture Highlights
- **FireMobileResidual blocks**: Combine squeeze-excitation, depthwise separable convolutions, and residual connections
- **Mixed convolution strategies**: 1×1 bottlenecks + 3×3 depthwise layers
- **Efficient design**: Suitable for deployment with low-precision (INT8) quantization

### Objectives
1. Train a hybrid CNN on Tiny ImageNet (64×64 images, 200 classes)
2. Quantize model weights and activations to INT8
3. Measure accuracy drop and model size compression
4. Track metrics throughout training and quantization

### Dataset
- **Tiny ImageNet-200**: 64×64 RGB images, 200 object classes
- **Training**: 90% of data (~82K images), **Validation**: 10% (~9K images)

### Expected Outputs
- FP32 baseline model with best validation accuracy
- INT8 quantized model via QAT
- Side-by-side accuracy and size comparison


## Section 1: Configuration & Environment

In [ ]:
import os
import random
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.ao.quantization as tq
import torchinfo

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    find_fuse_groups,
    load_best_model, build_qat, convert_to_int8,
    create_results_summary, disk_mb,
)
from models import TinyHybridNet, build_tinyhybridnet

import warnings
warnings.filterwarnings("ignore")
print("✓ All imports successful")

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.quantized.engine = "fbgemm"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = Path("./saved_models_hybrid_qat")

# ─── Hyperparameters — edit these ────────────────────────────────────────────
data_cfg = DataConfig(
    dataset_path="",           # filled after kagglehub download below
    batch_size=64,
    num_workers=0,             # 0 for Jupyter stability (no persistent_workers issue)
    seed=SEED,
)
fp32_cfg = TrainerConfig(
    epochs=30,
    lr=3e-4,
    weight_decay=5e-4,
    label_smoothing=0.1,
    use_amp=False,
    early_stopping_patience=None,
)
qat_extra = QATConfig(
    epochs=5,
    lr=1e-4,
    weight_decay=5e-4,
    freeze_bn_epoch=3,
    disable_observer_epoch=4,
)
# ─────────────────────────────────────────────────────────────────────────────
print(f"Device: {device}")

## Section 2: Dataset & Data Loading

**Preprocessing Strategy:**
- **Training**: RandomResizedCrop (0.6-1.0 scale), HorizontalFlip, ColorJitter, Normalization
- **Validation**: Resize, CenterCrop, Normalization (ImageNet stats)
- **Split**: 90/10 train/val with fixed seed for reproducibility

In [ ]:
import kagglehub

print("Downloading Tiny ImageNet-200...")
dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print(f"✓ Dataset available at: {train_path}")

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg, persistent_workers=False)

print(f"\n✓ Dataset loaded:")
print(f"  Training samples: {len(train_ds)}")
print(f"  Validation samples: {len(val_ds)}")
print(f"  Total classes: {data_cfg.num_classes}")
print(f"  Batch size: {data_cfg.batch_size}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Section 3: Model Definition

### TinyHybridNet Architecture
- **Stem**: 3 → 32 channels (3×3 conv)
- **FireMobileResidual blocks**: Combine depthwise separable + bottleneck + residuals
  - 1×1 squeeze layer reduces channel count by 75%
  - 3×3 depthwise convolution for spatial processing
  - 1×1 expansion back to output channels
  - Skip connection with identity/projection layer
- **Output**: AdaptiveAvgPool → Linear classifier
- **Quantization stubs**: QuantStub/DeQuantStub for QAT support

### Design Rationale
- Depthwise separable layers reduce FLOPs (~9× vs standard conv for 3×3)
- Squeeze-excitation improves parameter efficiency
- Residual connections improve gradient flow and model expressiveness

In [ ]:
# TinyHybridNet imported from models/tinyhybridnet.py (QAT-compatible version:
# inplace=False ReLU everywhere, FloatFunctional for residual adds)

MODEL_REGISTRY.clear()
register_model(
    "tinyhybridnet",
    build_tinyhybridnet,
    fuse_map=find_fuse_groups(build_tinyhybridnet()),
)

model = TinyHybridNet(num_classes=data_cfg.num_classes).to(device)
print("✓ TinyHybridNet created")
info = torchinfo.summary(model, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
print(info)

## Section 4: Training (FP32 Baseline)

### FP32 Training Strategy
- **Optimizer**: AdamW with learning rate decay
- **Scheduler**: Cosine annealing for smooth learning rate decay
- **Loss**: CrossEntropyLoss with label smoothing (0.1)
- **Epochs**: 30 epochs to ensure convergence
- **Checkpoint**: Save best model based on validation accuracy

In [ ]:
trainer = Trainer(
    model, train_loader, val_loader, fp32_cfg,
    device, SAVE_DIR, "tinyhybridnet", num_classes=data_cfg.num_classes,
)
fp32_results = trainer.fit()
best_fp32_acc = fp32_results["best_val_accuracy"]
print(f"\n✓ FP32 Training complete. Best validation accuracy: {best_fp32_acc:.2f}%")

In [ ]:
# Evaluate best FP32 checkpoint
fp32_model = load_best_model("tinyhybridnet", build_tinyhybridnet, SAVE_DIR, device)
fp32_eval = Trainer(fp32_model, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                    device, SAVE_DIR, "tinyhybridnet", num_classes=data_cfg.num_classes)
fp32_metrics = fp32_eval.evaluate(topk=(1, 5))
fp32_acc, fp32_loss = fp32_metrics["top1"], fp32_metrics["loss"]
print(f"FP32 best | top1={fp32_acc:.2f}% | loss={fp32_loss:.4f}")

## Section 5: Quantization-Aware Training (QAT)

### QAT Process
1. **Copy trained model**: Deep copy of best FP32 checkpoint
2. **Apply QAT config**: Set quantization configuration to FBGEMM backend
3. **Prepare for QAT**: Insert fake quantization modules into model
4. **Fine-tune**: Train for few epochs (~5) with frozen BN stats
5. **Convert to INT8**: Replace fake quant with actual INT8 kernels

In [ ]:
qat_model = build_qat("tinyhybridnet", save_dir=SAVE_DIR, device=device)
print("✓ QAT model prepared")
print(f"  Backend: {torch.backends.quantized.engine}")

In [ ]:
x, y = next(iter(train_loader))
print(f"Batch OK: inputs={tuple(x.shape)}  labels={tuple(y.shape)}")

In [ ]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_extra.epochs,
    lr=qat_extra.lr,
    weight_decay=qat_extra.weight_decay,
    use_amp=False,
)
cb = make_qat_callback(qat_extra.freeze_bn_epoch, qat_extra.disable_observer_epoch)

qat_trainer = Trainer(
    qat_model, train_loader, val_loader, qat_train_cfg,
    device, SAVE_DIR, "qat_tinyhybridnet", num_classes=data_cfg.num_classes,
    epoch_callback=cb,
)
qat_results = qat_trainer.fit()
best_qat_acc = qat_results["best_val_accuracy"]
print(f"\n✓ QAT complete | Best validation accuracy: {best_qat_acc:.2f}%")

## Section 6: Convert to INT8 & Export

In [ ]:
# Reload best QAT weights and convert to INT8
qat_best = torch.load(SAVE_DIR / "qat_tinyhybridnet_best.pth", map_location="cpu", weights_only=True)
qat_model.cpu()
qat_model.load_state_dict(qat_best["model_state_dict"])

int8_model = convert_to_int8(qat_model)
print("✓ Model converted to INT8")

int8_state_path = SAVE_DIR / "tinyhybrid_int8.pth"
torch.save(int8_model.state_dict(), int8_state_path)
print(f"✓ INT8 state dict saved: {int8_state_path}")

ts_path = SAVE_DIR / "tinyhybrid_int8_ts.pt"
traced = torch.jit.trace(int8_model.eval(), torch.randn(1, 3, data_cfg.img_size, data_cfg.img_size))
traced.save(str(ts_path))
print(f"✓ TorchScript INT8 model saved: {ts_path}")

## Section 7: Evaluation & Results Analysis

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
int8_eval = Trainer(int8_model, val_loader_cpu, val_loader_cpu, replace(fp32_cfg, use_amp=False),
                    torch.device("cpu"), SAVE_DIR, "int8", num_classes=data_cfg.num_classes)
int8_metrics = int8_eval.evaluate(topk=(1, 5))
qat_acc = int8_metrics["top1"]
qat_loss = int8_metrics["loss"]

print(f"\nINT8 Model:")
print(f"  Top-1 Accuracy: {qat_acc:.2f}%")
print(f"  Loss: {qat_loss:.4f}")

In [ ]:
# Summary comparison (single-worker, deterministic)
print("-" * 70)
print(f"FP32  | top1={fp32_acc:.2f}%  loss={fp32_loss:.4f}")
print(f"INT8  | top1={qat_acc:.2f}%  loss={qat_loss:.4f}")
print(f"Drop  | {fp32_acc - qat_acc:.2f}%")

In [ ]:
fp32_size = disk_mb(SAVE_DIR / "tinyhybridnet_best.pth")
int8_size = disk_mb(int8_state_path)
compression = fp32_size / int8_size if int8_size > 0 else float("nan")
info = torchinfo.summary(fp32_model, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)

print("=" * 70)
print("MODEL SIZE COMPARISON")
print("=" * 70)
print(f"Parameter Count: {info.total_params/1e6:.2f}M")
print(f"FP32 size:  {fp32_size:.2f} MB")
print(f"INT8 size:  {int8_size:.2f} MB")
print(f"Compression: {compression:.2f}× smaller  |  {100*(1-int8_size/fp32_size):.1f}% reduction")
print(f"\nAccuracy drop: {fp32_acc - qat_acc:.2f}% ({fp32_acc:.2f}% → {qat_acc:.2f}%)")

create_results_summary(
    results={
        "fp32": {"top1": fp32_acc, "loss": fp32_loss, "size_mb": fp32_size},
        "int8": {"top1": qat_acc, "loss": qat_loss, "size_mb": int8_size},
        "quantization_metrics": {
            "accuracy_drop": fp32_acc - qat_acc,
            "compression_ratio": compression,
        },
    },
    config=asdict(fp32_cfg),
    output_path=SAVE_DIR / "experiment_summary.json",
)
print(f"\n✓ Experiment summary saved: {SAVE_DIR / 'experiment_summary.json'}")